In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from dotenv import load_dotenv
from pathlib import Path
import os

In [ ]:
class TalkToDocument:
    def __init__(self):
        # Load environment variables. Ensure you have set-up an API key.
        load_dotenv()
        if not os.getenv("GEMINI_API_KEY"):
            raise ValueError("Missing GEMINI_API_KEY. Check your .env file setup.")

        self.llm_model_name = os.getenv("LLM_MODEL", "gemini-2.5-flash-lite")
        self.embeddings_model_name = os.getenv("EMBEDDINGS_MODEL", "models/gemini-embedding-001")

        # Initialize Google models.
        self.embeddings = GoogleGenerativeAIEmbeddings(model=self.embeddings_model_name)
        self.llm = ChatGoogleGenerativeAI(model=self.llm_model_name, temperature=0.1)

        self.index_path = "./leave_policy_faiss_index"

        # Placeholders for components built dynamically.
        self.vector_store = None
        self.retriever = None
        self.rag_chain = None

    def initialize_pipeline(self, file_path):
      # High-level method to orchestrate the entire set-up.
      # Checks if index already exists. If yes, loads it.
      if Path(self.index_path).exists():
          self.vector_store = FAISS.load_local(self.index_path, self.embeddings, allow_dangerous_deserialization=True)
      else:
          document_chunks = self.ingest_documents(file_path)
          self.build_vector_store(document_chunks)

      self.create_rag_chain()
      print("RAG pipeline is ready to query.")

    def ingest_documents(self, file_path):
        # Loads a text document and splits it into chunks.
        loader = TextLoader(file_path)
        documents = loader.load()

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
        document_chunks = text_splitter.split_documents(documents)
        return document_chunks

    def build_vector_store(self, document_chunks):
        # Creates a FAISS vector store from document chunks.
        self.vector_store = FAISS.from_documents(document_chunks, self.embeddings)
        self.vector_store.save_local(self.index_path)
        self.retriever = self.vector_store.as_retriever(search_kwargs={"k": 3})

    def create_rag_chain(self):
        # Helper to build a RAG chain. Defines the prompt template.
        system_prompt = ("You are a helpful assistant. Use the following retrieved context to answer user questions. If you don't know the answer, say I don't know. \n {context}")
        prompt = ChatPromptTemplate.from_messages([
                    ("system", system_prompt),
                    ("human", "{input}") ])
        question_answer_chain = create_stuff_documents_chain(self.llm, prompt)
        self.rag_chain = create_retrieval_chain(self.retriever, question_answer_chain)

    def query(self,query):
        # Queries the RAG pipeline and returns the response.
        return self.rag_chain.invoke({"input": query})

if __name__ == "__main__":
    # Instantiate the class.
    rag_system = TalkToDocument()

    # Ingest data and set-up the RAG chain. Pass the path to any local file you want to chat with.
    rag_system.initialize_pipeline("employee_leave_policy.txt")

In [ ]:
# Ask a question.
query = input("Please enter your query here : ")
response = rag_system.query(query)
print("AI assistant : ", response["answer"])

Please enter your query here : How many paid leaves are entitled each calender year??
AI assistant :  Based on the information provided, you are entitled to:

*   **15 days** per calendar year for vacations and personal days.
*   **7 days** of fully paid sick leave each year.


In [ ]:
# Ask a question.
query = input("Please enter your query here : ")
response = rag_system.query(query)
print("AI assistant : ", response["answer"])

Please enter your query here : Can I request a paid leave 2 days in advance?
AI assistant :  No, you cannot request a paid leave 2 days in advance. All leave must be submitted at least 2 weeks in advance.


In [ ]:
# Ask a question.
query = input("Please enter your query here : ")
response = rag_system.query(query)
print("AI assistant : ", response["answer"])

Please enter your query here : Do leaves get carried over to next year? If so, how many?
AI assistant :  Yes, leaves can be carried over to the next year. Up to 5 days of unused PTO may be rolled over.
